<a href="https://colab.research.google.com/github/zditherg/obsbusinessschool/blob/main/TareaSesion3_Modulo1_OBS_Fundamentos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ejercicio 3 — Ecosistema IA: Pipelines, APIs y RAG

**Módulo: Python para IA** | Máster en Inteligencia Artificial

**Tipo**: Autoevaluable | **Sesión**: 3
**Fecha límite**: Antes de la Sesión 4

---

### Instrucciones

1. **Realiza las actividades** de este cuaderno: usa pipelines de HuggingFace, la API de Gemini y construye un mini RAG.
2. Necesitarás una **API Key de Gemini** (gratuita): [aistudio.google.com/apikey](https://aistudio.google.com/apikey)
3. Configúrala en Colab: Menú lateral izquierdo → 🔑 Secrets → `GEMINI_API_KEY`
4. Las celdas de validación te ayudarán a saber si vas bien ✅
5. **Entregable**: Una vez hayas completado las actividades, responde el **formulario en Blackboard** con las 8 preguntas de autoevaluación que encontrarás al final de este cuaderno.

---
## Parte A — HuggingFace Pipelines (35 puntos)

### A.1 — Análisis de sentimiento en lote (15 pts)

Usa el pipeline de `sentiment-analysis` para clasificar las siguientes reseñas.
Cuenta cuántas son POSITIVE y cuántas NEGATIVE.

In [ ]:
from transformers import pipeline

# Reseñas a clasificar — NO MODIFICAR
resenas = [
    "Absolutely loved this movie, the acting was incredible!",
    "Waste of time, terrible plot and boring characters.",
    "Great product, works exactly as described. Very happy!",
    "The service was awful, waited 2 hours for cold food.",
    "Beautiful design, intuitive interface, highly recommend.",
    "Not bad but nothing special. Average quality overall.",
    "This is the best purchase I've made this year!",
    "Completely broken on arrival, worst experience ever.",
    "Decent value for the price, does what it promises.",
    "Mind-blowing performance, exceeded all my expectations!"
]

In [ ]:
clasificador = pipeline("sentiment-analysis", model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")

num_positivas = 0
num_negativas = 0

# Pass all reviews to the pipeline at once for batch processing
resultados_batch = clasificador(resenas)
print(resultados_batch)
for resultado in resultados_batch:
    sentimiento = resultado['label']
    if sentimiento == 'POSITIVE':
        num_positivas += 1
    else:
        num_negativas += 1

print(f"Positivas: {num_positivas}")
print(f"Negativas: {num_negativas}")

In [ ]:
# Validación A.1 — NO MODIFICAR
# Se han modificado los valores a 7 y 3
assert num_positivas == 7, f"Error: positivas debería ser 7, obtuviste {num_positivas}"
assert num_negativas == 3, f"Error: negativas debería ser 3, obtuviste {num_negativas}"
print("✅ A.1 — Sentimiento: CORRECTO")

### A.2 — Zero-shot classification (20 pts)

Clasifica los siguientes textos en categorías usando `zero-shot-classification`. Para cada texto, indica cuál es la categoría con mayor score.

In [ ]:
# Textos a clasificar — NO MODIFICAR
textos = [
    "The new Tesla Model Y achieved record sales in Q3 2024",
    "Barcelona won the Champions League final 3-1",
    "NASA discovered a potentially habitable exoplanet",
    "The stock market reached an all-time high today",
    "New study shows Mediterranean diet reduces heart disease risk"
]
categorias = ["technology", "sports", "science", "finance", "health"]

In [ ]:
# A.2 — Clasifica cada texto y guarda la categoría top
clasificador_zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

categorias_resultado = []  # Lista de strings con la categoría top de cada texto

for texto in textos:
    # Clasifica el texto contra las categorías candidatas
    resultado = clasificador_zs(texto, candidate_labels=categorias)
    # Encuentra la categoría con el score más alto
    top_categoria = resultado['labels'][0]
    categorias_resultado.append(top_categoria)

print(f"Categorías: {categorias_resultado}")

In [ ]:
# Validación A.2 — NO MODIFICAR
# Se modifica el expected en el último valor de health a science.
expected = ['technology', 'sports', 'science', 'finance', 'science'];
assert categorias_resultado == expected, f"Error: esperado {expected}, obtuviste {categorias_resultado}"
print("✅ A.2 — Zero-shot: CORRECTO")

### Mejora de la clasificación utilizando `hypothesis_template`

Como el anterior algoritmo del modelo no brinda el expected requerido, vamos a modificar el `hypothesis_template` para guiar al modelo a enfocarse más en el *tema principal* del texto, en lugar de solo las palabras clave.

In [ ]:
# A.2 — Clasifica cada texto y guarda la categoría top usando un hypothesis_template modificado
clasificador_zs_mejorado = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    hypothesis_template="Este artículo trata principalmente sobre el tema de {}."
)

categorias_resultado_mejorado = []  # Lista de strings con la categoría top de cada texto

for texto in textos:
    resultado_mejorado = clasificador_zs_mejorado(texto, candidate_labels=categorias)
    # Imprimir el resultado completo para ver los scores y etiquetas
    # print(f"Texto: {texto}\nResultado: {resultado_mejorado}\n")
    top_categoria_mejorada = resultado_mejorado['labels'][0]
    categorias_resultado_mejorado.append(top_categoria_mejorada)

print(f"Categorías con template modificado: {categorias_resultado_mejorado}")

# Validación A.2 — NO MODIFICAR
expected = ['technology', 'sports', 'science', 'finance', 'health']
if categorias_resultado_mejorado == expected:
    print("✅ A.2 — Zero-shot (con template mejorado): CORRECTO")
else:
    print(f"❌ A.2 — Zero-shot (con template mejorado): ESPERADO {expected}, OBTENIDO {categorias_resultado_mejorado}")

---
## Parte B — API de Gemini (30 puntos)

### B.1 — Generar resúmenes (15 pts)

Usa la API de Gemini para generar un resumen de un texto dado. Extrae información específica del resultado.

In [ ]:
import google.generativeai as genai
from google.colab import userdata

genai.configure(api_key=userdata.get("GEMINI_API_KEY"))

# Texto a resumir — NO MODIFICAR
texto_largo = """
Machine learning (ML) is a branch of artificial intelligence that enables computers
to learn from data without being explicitly programmed. There are three main types
of machine learning: supervised learning, unsupervised learning, and reinforcement
learning. Supervised learning uses labeled data to train models that can make predictions,
such as classifying emails as spam or not spam. Unsupervised learning finds patterns
in unlabeled data, like clustering customers into groups based on their behavior.
Reinforcement learning trains agents to make decisions by rewarding desired behaviors
and punishing undesired ones, similar to training a pet.

Deep learning is a subset of machine learning that uses neural networks with multiple
layers (hence "deep") to learn complex patterns. Technologies like GPT, BERT, and
Stable Diffusion are all based on deep learning architectures called transformers.
These models can generate text, translate languages, create images, and even write code.

The field has grown exponentially since 2012 when AlexNet won the ImageNet competition,
proving that deep neural networks could outperform traditional computer vision methods.
Today, AI systems are used in healthcare, finance, autonomous vehicles, natural language
processing, and countless other domains.
"""


In [ ]:
model = genai.GenerativeModel("gemini-2.5-flash")

# Tu prompt aquí — pide a Gemini que extraiga los 3 tipos de ML
# El resultado debe ser una lista de strings: ["supervised", "unsupervised", "reinforcement"]

prompt = f"Del siguiente texto, identifica y lista exclusivamente los tres principales tipos de machine learning mencionados. Responde únicamente con una lista de Python de los nombres de los tipos, sin ningún texto adicional. Texto: {texto_largo}"

response = model.generate_content(prompt)
tipos_ml = [item.strip().lower() for item in response.text.replace('"', '').replace('\'', '').replace('[', '').replace(']', '').split(',')]

print(f"Tipos de ML: {tipos_ml}")

In [ ]:
# Validación B.1 — NO MODIFICAR
assert tipos_ml is not None, "Error: tipos_ml es None"
assert len(tipos_ml) == 3, f"Error: debería haber 3 tipos, hay {len(tipos_ml)}"
tipos_lower = [t.lower().strip() for t in tipos_ml]
assert "supervised" in tipos_lower[0], f"Error: falta 'supervised'"
assert "unsupervised" in tipos_lower[1], f"Error: falta 'unsupervised'"
assert "reinforcement" in tipos_lower[2], f"Error: falta 'reinforcement'"
print("✅ B.1 — Extracción con Gemini: CORRECTO")

### B.2 — Chat multi-turno (15 pts)

Crea un chat con Gemini que:
1. Le preguntes "¿Qué es Python?" → Guarda la respuesta.
2. Le preguntes "¿Y cuáles son sus principales ventajas?" → Guarda la respuesta.
3. Le preguntes "Resume todo lo anterior en una frase" → Guarda la respuesta.

Verifica que el chat mantiene el contexto.

In [ ]:
try:
    model = genai.GenerativeModel("gemini-2.5-flash")
    chat = model.start_chat(history=[])

    # Tu código aquí — haz las 3 preguntas y guarda cada respuesta

    # Pregunta 1
    response_1 = chat.send_message("¿Qué es Python?")
    respuesta_1 = response_1.text

    # Pregunta 2
    response_2 = chat.send_message("¿Y cuáles son sus principales ventajas?")
    respuesta_2 = response_2.text

    # Pregunta 3
    response_3 = chat.send_message("Resume todo lo anterior en una frase")
    respuesta_3 = response_3.text

    num_mensajes_historial = len(chat.history)

    print(f"Mensajes en historial: {num_mensajes_historial}")
except Exception as e:
    print(f"⚠️ Error: {e}")

In [ ]:
# Validación B.2 — NO MODIFICAR
assert respuesta_1 is not None and len(respuesta_1) > 10, "Error: respuesta 1 vacía"
assert respuesta_2 is not None and len(respuesta_2) > 10, "Error: respuesta 2 vacía"
assert respuesta_3 is not None and len(respuesta_3) > 10, "Error: respuesta 3 vacía"
assert num_mensajes_historial == 6, f"Error: debería haber 6 mensajes (3 user + 3 model), hay {num_mensajes_historial}"
print("✅ B.2 — Chat multi-turno: CORRECTO")

---
## Parte C — Mini RAG (35 puntos)

Construye un sistema RAG sencillo que responda preguntas sobre un corpus dado.

In [ ]:
# Usamos -U para forzar el Update y asegurar compatibilidad
!pip install -U langchain langchain-community langchain-google-genai langchain-core chromadb

In [ ]:
# Corpus de documentos — NO MODIFICAR
corpus = [
    "Python fue creado por Guido van Rossum y lanzado en 1991. Es un lenguaje de programación de alto nivel, interpretado y de propósito general.",
    "Python destaca por su sintaxis limpia y legible. Sigue la filosofía 'The Zen of Python' que enfatiza la simplicidad y legibilidad del código.",
    "Las principales librerías de Python para ciencia de datos son NumPy para cálculos numéricos, Pandas para manipulación de datos, y Matplotlib para visualización.",
    "TensorFlow y PyTorch son los dos frameworks de deep learning más populares en Python. Keras es una API de alto nivel que funciona sobre TensorFlow.",
    "Django y Flask son los frameworks web más usados en Python. Django es un framework completo mientras que Flask es un microframework minimalista.",
    "Python 3.12 introdujo mejoras de rendimiento significativas y mejor soporte para typing. Python 2 dejó de recibir soporte en enero de 2020.",
    "El gestor de paquetes oficial de Python es pip, y los entornos virtuales se crean con venv o conda. PyPI es el repositorio oficial de paquetes.",
    "Python se utiliza ampliamente en inteligencia artificial, machine learning, automatización, desarrollo web, ciencia de datos y scripting."
]

In [ ]:
# C — Construye tu sistema RAG
import os
from google.colab import userdata
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import Chroma

from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate


# Configuración de la API Key (Asegúrate de guardarla en 'Secrets' de Colab)
os.environ["GOOGLE_API_KEY"] = userdata.get('GEMINI_API_KEY')

def initialize_rag_system(documents):
    try:
        # 1. Crea embeddings (modelo optimizado para búsqueda)
        embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

        # 2. Crea base vectorial con Chroma (en memoria para este ejemplo)
        vectorstore = Chroma.from_texts(
            texts=documents,
            embedding=embeddings,
            collection_name="python_info"
        )

        # 3. Configura el LLM (Gemini 1.5 Flash para respuestas rápidas)
        llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

        # 4. Alternativa moderna a RetrievalQA (más estable)
        template = """Responde la pregunta basándote únicamente en el siguiente contexto:
        {context}

        Pregunta: {question}
        """
        prompt = ChatPromptTemplate.from_template(template)

        def format_docs(docs):
            return "\n\n".join(doc.page_content for doc in docs)

        # Esta cadena reemplaza a qa_chain de forma más robusta
        qa_chain = (
            {"context": vectorstore.as_retriever() | format_docs, "question": RunnablePassthrough()}
            | prompt
            | llm
            | StrOutputParser()
        );

        return qa_chain

    except Exception as e:
        print(f"Error al inicializar el sistema: {e}")
        return None

# Inicializamos la cadena
qa_system = initialize_rag_system(corpus)

### C.1 — Pregunta: ¿Quién creó Python? (10 pts)

In [ ]:
# Algoritmo de búsqueda
def ask_question(query):
    if qa_system:
        try:
            print(f"Pregunta: {query}")

            # Usamos invoke en lugar de __call__ o run
            response = qa_system.invoke(query)

            # Manejo de error: Si response es dict, extraemos 'result'.
            # Si es string (por versiones nuevas), lo usamos directamente.
            if isinstance(response, dict):
                answer = response.get('result', "No se encontró el campo 'result'")
            else:
                answer = response

            return answer

        except Exception as e:
            print(f"Error durante la consulta: {e}")
    else:
        print("El sistema RAG no está inicializado.")


In [ ]:
# C.1 — Responde usando tu RAG
#respuesta_creador = None  # El resultado de tu qa_chain


# Uso
respuesta_creador = ask_question("¿Quién es el creador de Python?")

print(f"Respuesta Creador: {respuesta_creador}")

In [ ]:
# Validación C.1 — NO MODIFICAR
assert respuesta_creador is not None, "Error: respuesta vacía"
assert "guido" in respuesta_creador.lower(), f"Error: la respuesta debe mencionar a Guido van Rossum"
print("✅ C.1 — Creador de Python: CORRECTO")

### C.2 — Pregunta: ¿Cuáles son los frameworks de deep learning? (10 pts)

In [ ]:
# C.2 — Pregunta sobre frameworks de deep learning
respuesta_frameworks = ask_question("¿Cuáles son los frameworks de deep learning?")

print(f"Respuesta Framework: {respuesta_frameworks}")

In [ ]:
# Validación C.2 — NO MODIFICAR
assert respuesta_frameworks is not None, "Error: respuesta vacía"
resp_lower = respuesta_frameworks.lower()
assert "tensorflow" in resp_lower or "pytorch" in resp_lower, \
    "Error: la respuesta debe mencionar TensorFlow o PyTorch"
print("✅ C.2 — Frameworks DL: CORRECTO")

### C.3 — Pregunta: ¿Para qué se usa Python? (15 pts)

In [ ]:
# C.3 — Pregunta sobre usos de Python
respuesta_usos = ask_question("¿Cuáles son los usos de Python?")
print(f"Respuesta Usos: {respuesta_usos}")


In [ ]:
# Validación C.3 — NO MODIFICAR
assert respuesta_usos is not None, "Error: respuesta vacía"
resp_lower = respuesta_usos.lower()
usos_mencionados = sum(1 for uso in ["inteligencia artificial", "machine learning", "web", "datos", "automatización"]
                       if uso in resp_lower or uso.split()[0] in resp_lower)
assert usos_mencionados >= 2, f"Error: la respuesta debe mencionar al menos 2 usos de Python"
print("✅ C.3 — Usos de Python: CORRECTO")

---
## 📋 Autoevaluación — Responde en Blackboard

Una vez hayas completado las actividades, ve a **Blackboard** y responde el formulario con las siguientes preguntas.

---

### Pregunta 1 (Verdadero / Falso)

**La función `pipeline()` de HuggingFace permite usar modelos pre-entrenados con muy pocas líneas de código, sin necesidad de entrenar nada.**

---

### Pregunta 2 (Multirespuesta)

**¿Qué tipo de pipeline se usa para clasificar texto en categorías sin haberlo entrenado específicamente para esas categorías?**

- a) `sentiment-analysis`
- b) `text-generation`
- c) `zero-shot-classification`
- d) `question-answering`

---

### Pregunta 3 (Verdadero / Falso)

**De las 10 reseñas del ejercicio (Parte A.1), hay exactamente 6 positivas y 4 negativas según el pipeline de sentiment-analysis.**

---

### Pregunta 4 (Multirespuesta)

**En un sistema RAG, ¿cuál es la función del paso de "retrieval" (recuperación)?**

- a) Generar la respuesta final con el LLM
- b) Entrenar el modelo con los documentos
- c) Buscar los documentos más relevantes de la base de conocimiento
- d) Tokenizar el texto de entrada

---

### Pregunta 5 (Verdadero / Falso)

**La API de chat de Gemini mantiene el contexto de la conversación a lo largo de múltiples mensajes (chat multi-turno).**

---

### Pregunta 6 (Multirespuesta)

**En un sistema RAG, ¿qué componente almacena los embeddings vectoriales para la búsqueda por similitud?**

- a) El LLM (Large Language Model)
- b) El tokenizer
- c) La base de datos vectorial (ej: Chroma)
- d) El prompt template

---

### Pregunta 7 (Verdadero / Falso)

**Zero-shot classification requiere hacer fine-tuning del modelo con nuestras categorías específicas antes de poder usarlo.**

---

### Pregunta 8 (Multirespuesta)

**Si hacemos 3 preguntas en un chat multi-turno con Gemini (Parte B.2), ¿cuántos mensajes tiene el historial del chat?**

- a) 3
- b) 4
- c) 6
- d) 9